In [11]:
from device_python.esp32_device_bt import ESP32DeviceBT

await ESP32DeviceBT.scan_devices()

[BLEDevice(2276FE3C-DECA-FC25-D5C1-A960D88DA29C, MIKATSIIKI)]

In [9]:

from bleak import BleakScanner

device = await BleakScanner.find_device_by_name("MIKATSIIKI", timeout=0.1)
device

BLEDevice(2276FE3C-DECA-FC25-D5C1-A960D88DA29C, MIKATSIIKI)

In [58]:
from bleak import BleakClient

assert device is not None
client = BleakClient(device.address)
await client.connect()

In [73]:
await client.disconnect()

In [59]:
for service in client.services:
    print(f"service {service.uuid}")
    for char in service.characteristics:
        print(f"  char {char.uuid}")
        print(f"    props {char.properties}")
        for desc in char.descriptors:
            print(f"    desc {desc.uuid} {desc.description}")

service 0000abf0-0000-1000-8000-00805f9b34fb
  char 0000abf1-0000-1000-8000-00805f9b34fb
    props ['write-without-response', 'notify']
    desc 00002902-0000-1000-8000-00805f9b34fb Client Characteristic Configuration


In [60]:
char = client.services.get_characteristic("ABF1")
assert char is not None
assert 'write-without-response' in char.properties
assert 'notify' in char.properties

def notification_handler(sender, data: bytearray):
    # sender 是 characteristic 的 uuid
    print(f"收到来自 {sender} 的数据: {data}")

await client.start_notify(char.uuid,notification_handler)


In [72]:
import json
import time
import zlib
from uuid import uuid4

assert char is not None

request_id = str(uuid4())
command = {
    "name": "system_state_get",
    "parameters": {},
    "request_id": request_id,
}
command_str = json.dumps(command, separators=(",", ":"))
print(f"esp32 > {command_str}")
start_time_ns = time.perf_counter_ns()
serial_line = command_str.encode()
serial_line += (
    zlib.crc32(serial_line)
    .to_bytes(length=4, byteorder="little", signed=False)
    .hex()
    .upper()
    .encode()
)
serial_line += b"\n"

await client.write_gatt_char(char.uuid, serial_line, response=False)



esp32 > {"name":"system_state_get","parameters":{},"request_id":"b4f36ce4-89e1-45af-90bf-a584c0027a3f"}


收到来自 0000abf1-0000-1000-8000-00805f9b34fb (Handle: 41): Vendor specific 的数据: bytearray(b'{"type":"response","result":{"memory":{"internal":{"integrity":true,"total_size":384651,"free_bytes":238355,"allocated_bytes":131936,"largest_free_block":163824,"minimum_free_bytes":201407,"allocated_blocks":496,"free_blocks":10,"total_blocks":506},"spiram":{"integrity":true,"total_size":8388608,"free_bytes":8375320,"allocated_bytes":4592,"largest_free_block":8257520,"minimum_free_bytes":8356792,"allocated_blocks":319,"free_blocks":5,"total_blocks":324}},"total_run_time_s":112941.774093,"tasks":[{"task_name')
收到来自 0000abf1-0000-1000-8000-00805f9b34fb (Handle: 41): Vendor specific 的数据: bytearray(b'":"cmd_bt","task_number":13,"current_state":"Running","current_priority":10,"base_priority":10,"run_time_s":147.288699,"run_time_ratio":0.0013041117884222148,"stack_high_water_mark":6700,"core_id":1,"peak_heap_usage":36168,"current_heap_usage":4648,"heap_count":1,"heaps":[{"name":"RAM","caps":1071118,"size